# NLST Sybil Tumor Imaging Benchmark Analysis

This notebook performs the analysis for the NLST Sybil case study, comparing Lung Stage Classification and Histology Classification.
It generates performance plots (Accuracy, AUC), ROC curves, and analyzes feature similarity using Mutual k-Nearest Neighbors Overlap.


In [ ]:
import os
import sys
import numpy as np
import pandas as pd
import pickle
import plotly.graph_objects as go
import plotly.express as px
import matplotlib.pyplot as plt
from pathlib import Path
from sklearn.neighbors import NearestNeighbors
from sklearn.metrics import roc_curve, auc

# Add modelling directory to path to import utils
# This assumes the script is run from notebooks/case_study
sys.path.append(str(Path("../../modelling").resolve()))

# Configuration
models = [
    "CTClipVit", "CTFM", "FMCIB", "Merlin", "ModelsGen", 
    "PASTA", "SUPREME", "VISTA3D", "Voco"
]

base_dir = Path(".")
de_stag_dir = base_dir / "de_stag"
de_type_dir = base_dir / "de_type"

# Ensure output directories exist
(base_dir / "metrics").mkdir(parents=True, exist_ok=True)
(de_stag_dir / "metrics" / "roc").mkdir(parents=True, exist_ok=True)
(de_type_dir / "metrics" / "roc").mkdir(parents=True, exist_ok=True)

def apply_aggregation_filter(v, model_name):
    """Aggregation filter for different models."""
    # If features are already aggregated (e.g. shape (1, D)), return as is
    if v.ndim <= 2:
        return v
        
    if model_name == "MedImageInsightExtractor":
        return v.mean(axis=0)
    elif model_name == "CTClipVitExtractor":
        return v.mean(axis=(1,2,3))
    elif model_name == "PASTAExtractor":
        return v.mean(axis=(2,3,4))        
    else:
        return v

def compute_knn_indices(model_features, num_neighbors=10, metric="cosine"):
    model_neighbors = {}
    for model_name, features in model_features.items():
        nn_model = NearestNeighbors(n_neighbors=num_neighbors + 1, metric=metric)
        nn_model.fit(features)
        _, indices = nn_model.kneighbors(features)
        model_neighbors[model_name] = indices[:, 1:]
    return model_neighbors

def compute_overlap_matrix(model_neighbors):
    model_list = list(model_neighbors.keys())
    n_models = len(model_list)
    overlap_matrix = np.full((n_models, n_models), np.nan)
    for i in range(n_models):
        neighbors_a = model_neighbors[model_list[i]]
        for j in range(i + 1, n_models):
            neighbors_b = model_neighbors[model_list[j]]
            if neighbors_a.shape[0] != neighbors_b.shape[0]:
                continue
            common_flags = (neighbors_a[:, :, None] == neighbors_b[:, None, :]).any(axis=2)
            sample_overlaps = np.sum(common_flags, axis=1)
            avg_overlap = np.mean(sample_overlaps)
            overlap_matrix[i, j] = avg_overlap
            overlap_matrix[j, i] = avg_overlap
    np.fill_diagonal(overlap_matrix, 0)
    overlap_matrix = np.nan_to_num(overlap_matrix, nan=0.0)
    return overlap_matrix, model_list

def plot_overlap_matrix(overlap_matrix, model_list, title="Mutual k-Nearest Neighbors Overlap Scores", width=1200, height=1200, color="Greens", tickangle=90, font_size=24):
    overlap_matrix = np.nan_to_num(overlap_matrix, nan=0.0)
    vmax = float(np.max(overlap_matrix)) if overlap_matrix.size else 1.0
    fig_size = (width / 100, height / 100)
    fig, ax = plt.subplots(figsize=fig_size)
    im = ax.imshow(overlap_matrix, cmap=color, vmin=0, vmax=max(vmax, 1e-6))
    
    # We want to show all ticks...
    ax.set_xticks(np.arange(len(model_list)))
    ax.set_yticks(np.arange(len(model_list)))
    
    ax.set_xticklabels(model_list, rotation=tickangle, ha="right", fontsize=font_size * 0.6)
    ax.set_yticklabels(model_list, fontsize=font_size * 0.6)
    ax.set_title(title, fontsize=font_size)
    ax.tick_params(length=6, width=1.5)
    cbar = fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
    cbar.ax.tick_params(labelsize=font_size * 0.6)
    fig.tight_layout()
    return fig

def apply_r_style(fig, font_size=20):
    """Apply a minimal ggplot-like style."""
    fig.update_layout(
        font=dict(size=font_size, family="DejaVu Serif"),
        plot_bgcolor="white",
        paper_bgcolor="white",
        xaxis=dict(
            showgrid=True,
            gridcolor="#D0D0D0",
            zeroline=False,
            linecolor="#4B4B4B",
            mirror=True,
            ticks="outside",
            ticklen=6,
            tickwidth=1.5,
        ),
        yaxis=dict(
            showgrid=True,
            gridcolor="#D0D0D0",
            zeroline=False,
            linecolor="#4B4B4B",
            mirror=True,
            ticks="outside",
            ticklen=6,
            tickwidth=1.5,
        ),
        margin=dict(l=60, r=40, t=60, b=60),
    )
    return fig

def load_scores(task_dir, models):
    results = {}
    for model in models:
        file_path = task_dir / "metrics" / "scores" / f"{model}_scores.npz"
        if file_path.exists():
            data = np.load(file_path)
            results[model] = {
                "test_y": data["test_y"],
                "test_pred": data["test_pred"],
                "test_auc": float(data["auc_values"][2]),
                "test_acc": float(data["test_accuracy_value"])
            }
    return results

def plot_metrics_separate(results, task_name, output_prefix):
    data = []
    for model in models:
        if model in results:
            data.append({
                "Model": model, 
                "Accuracy": results[model]["test_acc"], 
                "AUC": results[model]["test_auc"]
            })
    
    if not data:
        print(f"No data for {task_name}")
        return

    df = pd.DataFrame(data)
    
    # Define a light red color for the bars
    light_red = '#F7969C'

    for metric in ["Accuracy", "AUC"]:
        fig = px.bar(
            df, 
            x="Model", 
            y=metric, 
            title=f"{task_name} - {metric}",
            text_auto='.2f'
        )
        # Change color to uniform light red
        fig.update_traces(marker_color=light_red) 
        
        fig.update_layout(
            xaxis_title="",
            yaxis_title=metric,
            width=800,
            height=500,
            showlegend=False
        )
        apply_r_style(fig)
        
        # Save
        out_file = base_dir / "metrics" / f"{output_prefix}_{metric.lower()}.png"
        fig.write_image(str(out_file))
        print(f"Saved {out_file}")

def load_all_features(feature_dir, models):
    model_features = {}
    for model in models:
        pkl_path = feature_dir / f"{model}_features.pkl"
        if not pkl_path.exists():
            print(f"Feature file not found: {pkl_path}")
            continue
            
        try:
            with open(pkl_path, 'rb') as f:
                data = pickle.load(f)
            
            if 'all' in data:
                # Construct potential class name
                extractor_name = f"{model}Extractor"
                if model == "CTClipVit": extractor_name = "CTClipVitExtractor"
                if model == "PASTA": extractor_name = "PASTAExtractor"
                if model == "MedImageInsight": extractor_name = "MedImageInsightExtractor"
                
                processed_feats = []
                for item in data['all']:
                    f_in = item['feature'] 
                    f_out = apply_aggregation_filter(f_in, extractor_name)
                    processed_feats.append(f_out.flatten())
                    
                feats = np.vstack(processed_feats)
                model_features[model] = feats

        except Exception as e:
            print(f"Error loading {model}: {e}")

    return model_features

def plot_roc_minimalist(results, title, output_path):
    fig = go.Figure()
    fig.add_shape(
        type='line', line=dict(dash='dash', color='#D3D3D3'),
        x0=0, x1=1, y0=0, y1=1
    )

    colors = px.colors.qualitative.Safe
    
    for i, (model, data) in enumerate(results.items()):
        y_true = data["test_y"]
        y_score = data["test_pred"]
        if y_score.ndim > 1 and y_score.shape[1] > 1:
             y_score = y_score[:, 1]

        fpr, tpr, _ = roc_curve(y_true, y_score)
        auc_score = data["test_auc"]
        
        fig.add_trace(go.Scatter(
            x=fpr, y=tpr,
            name=f"{model} ({auc_score:.2f})",
            mode='lines',
            line=dict(width=2, color=colors[i % len(colors)])
        ))

    fig.update_layout(
        title=dict(text=title, x=0.5, font=dict(size=24)),
        xaxis_title='False Positive Rate',
        yaxis_title='True Positive Rate',
        width=800, height=600,
        legend=dict(x=0.65, y=0.05, bgcolor='rgba(255,255,255,0.8)', bordercolor="#E5E5E5", borderwidth=1),
        plot_bgcolor='white'
    )
    fig.update_xaxes(showgrid=True, gridcolor='#F0F0F0', zeroline=False, showline=True, linecolor='black')
    fig.update_yaxes(showgrid=True, gridcolor='#F0F0F0', zeroline=False, showline=True, linecolor='black')
    
    apply_r_style(fig)
    
    fig.write_image(str(output_path))
    print(f"Saved {output_path}")

if __name__ == "__main__":
    # Load Scores
    print("Loading scores...")
    results_stag = load_scores(de_stag_dir, models)
    results_type = load_scores(de_type_dir, models)

    # Plot Metrics
    print("Plotting metrics...")
    plot_metrics_separate(results_stag, "Lung Stage Classification", "de_stag")
    plot_metrics_separate(results_type, "Histology Classification", "de_type")

    # Plot ROCs
    print("Plotting ROCs...")
    plot_roc_minimalist(results_stag, "Lung Stage ROC", de_stag_dir / "metrics" / "roc" / "combined_roc_minimal.png")
    plot_roc_minimalist(results_type, "Histology ROC", de_type_dir / "metrics" / "roc" / "combined_roc_minimal.png")

    # Features Analysis
    print("Loading features and computing KNN overlap...")
    model_features = load_all_features(de_stag_dir / "features", models)
    if model_features:
        model_neighbors = compute_knn_indices(model_features, num_neighbors=10, metric="cosine")
        overlap_matrix, model_list = compute_overlap_matrix(model_neighbors)
        
        fig_overlap = plot_overlap_matrix(
            overlap_matrix, 
            model_list, 
            title="Mutual k-NN Overlap (Features)", 
            width=1000, 
            height=1000,
            color="Greens"
        )
        
        overlap_path = base_dir / "metrics" / "feature_knn_overlap.png"
        fig_overlap.savefig(str(overlap_path), bbox_inches='tight', dpi=300)
        print(f"Saved {overlap_path}")


Loading scores...
Plotting metrics...
Saved metrics/de_stag_accuracy.png
Saved metrics/de_stag_auc.png
Saved metrics/de_type_accuracy.png
Saved metrics/de_type_auc.png
Plotting ROCs...
Saved de_stag/metrics/roc/combined_roc_minimal.png
